# 任务 1: 数据预处理 / Task 1: Data Preprocessing

**Student Dropout and Academic Success Dataset**

本 Notebook 包含以下内容 / This notebook covers:
1. 加载与检查数据集 / Loading and inspecting the dataset
2. 缺失值处理 / Handling missing values
3. 分类特征编码 / Encoding categorical features
4. 数值特征标准化 / Standardizing numerical features
5. 数据模式观察与洞察总结 / Summary of observed patterns and insights

In [ ]:
import sys
import os

# 将项目根目录添加到路径，以便导入 src 模块
# Add project root to path so src can be imported
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 导入预处理工具函数 / Import preprocessing utilities
from src.preprocessing import (
    load_data,
    get_feature_types,
    handle_missing_values,
    encode_features,
    standardize_features,
    preprocess_pipeline,
)

# 导入数据下载工具，自动检测并下载数据集
# Import data fetcher — auto-downloads dataset if not present
from src.fetch_data import download_dataset

# 设置绘图风格 / Set plotting style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

## 1. 数据加载与检查 / Load and Inspect Data

In [ ]:
# 下载原始数据（若已存在则跳过）/ Download raw data (skipped if already present)
data_dir = os.path.join(project_root, "data")
download_dataset(data_dir=data_dir)

# 加载原始数据 / Load raw data
X, y = load_data(data_dir=data_dir)

In [ ]:
# 查看特征数据类型分布 / Check feature data type distribution
print("\n=== Feature Data Types / 特征数据类型 ===")
print(X.dtypes.value_counts())
print(f"\nTotal features / 总特征数: {X.shape[1]}")
print(f"Total samples / 总样本数: {X.shape[0]}")

In [ ]:
# 查看前 10 个特征的基本统计量 / Basic statistics for first 10 features
print("\n=== Basic Statistics / 基本统计量 (first 10 features) ===")
X.iloc[:, :10].describe().T

In [ ]:
# 查看剩余特征的基本统计量 / Basic statistics for remaining features
print("\n=== Basic Statistics / 基本统计量 (remaining features) ===")
X.iloc[:, 10:].describe().T

## 2. 特征类型分类 / Feature Type Classification

根据数据集的语义，特征被分为四类：
Features are classified into four groups based on their semantics:

- **Nominal / 名义型**: 无序类别（如婚姻状态、课程）
- **Binary / 二值型**: 0/1 指示变量（如性别、奖学金）
- **Count / 计数型**: 整数计数（如注册学分、通过学分）
- **Continuous / 连续型**: 浮点数值（如入学成绩、GDP）

In [ ]:
# 获取特征类型分类 / Get feature type classification
feature_types = get_feature_types(X)

print(f"Nominal features / 名义型特征 ({len(feature_types['nominal'])}):")
for f in feature_types["nominal"]:
    print(f"  - {f} (unique: {X[f].nunique()})")

print(f"\nBinary features / 二值型特征 ({len(feature_types['binary'])}):")
for f in feature_types["binary"]:
    print(f"  - {f}")

print(f"\nCount features / 计数型特征 ({len(feature_types['count'])}):")
for f in feature_types["count"]:
    print(f"  - {f}")

print(f"\nContinuous features / 连续型特征 ({len(feature_types['continuous'])}):")
for f in feature_types["continuous"]:
    print(f"  - {f}")

## 3. 目标变量分布 / Target Distribution

数据集包含三个类别：**Graduate（毕业）**（多数类）、**Dropout（退学）**、**Enrolled（在读）**。

In [ ]:
# 将数字标签映射为类别名称 / Map numeric labels to class names
target_names = {0: "Dropout", 1: "Enrolled", 2: "Graduate"}
y_named = y.map(target_names)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 柱状图 / Bar chart
counts = y_named.value_counts()
colors = ["#e74c3c", "#f39c12", "#2ecc71"]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor="white", linewidth=1.2)
axes[0].set_title("Target Class Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Count")
# 在柱顶标注数量和百分比 / Annotate count and percentage on top of bars
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                 f"{val}\n({val/len(y)*100:.1f}%)", ha="center", va="bottom", fontsize=10)

# 饼图 / Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors, autopct="%1.1f%%",
            startangle=90, textprops={"fontsize": 11})
axes[1].set_title("Target Class Proportions", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(project_root, "data", "plots", "01_target_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

**观察 / Observation**: 数据集存在**类别不平衡** — Graduate（49.9%）占主导，Enrolled（18.0%）比例最低。这可能影响模型训练和评估。

## 4. 缺失值分析 / Missing Value Analysis

In [ ]:
# 检查并处理缺失值 / Check and handle missing values
X_clean, missing_info = handle_missing_values(X, strategy="median")
print(f"\nMissing info / 缺失值信息: {missing_info}")

**结果 / Result**: 数据集中未检测到缺失值。这是一个完整的数据集。

## 5. 特征编码 / Feature Encoding

对名义型特征进行 one-hot 编码，避免引入人为的序数关系。
Nominal categorical features are one-hot encoded to avoid introducing artificial ordinal relationships.

In [ ]:
# 对名义型特征进行 one-hot 编码 / One-hot encode nominal features
X_encoded = encode_features(X_clean, feature_types, method="onehot")
print(f"Shape before encoding / 编码前维度: {X_clean.shape}")
print(f"Shape after encoding / 编码后维度:  {X_encoded.shape}")
print(f"New features added / 新增特征数: {X_encoded.shape[1] - X_clean.shape[1]}")

## 6. 特征标准化 / Feature Standardization

对连续型和计数型特征进行标准化（零均值、单位方差），二值型特征保持不变。
Continuous and count features are standardized to zero mean and unit variance. Binary features are kept as-is.

In [ ]:
# 标准化数值特征 / Standardize numerical features
X_scaled, scaler = standardize_features(X_encoded, feature_types, fit=True)
print(f"Final shape / 最终维度: {X_scaled.shape}")

In [ ]:
# 查看标准化后的统计量 / Check statistics after standardization
print("\n=== Standardized Features / 标准化后特征 (sample) ===")
scale_cols = feature_types["continuous"] + feature_types["count"]
scale_cols = [c for c in scale_cols if c in X_scaled.columns]
X_scaled[scale_cols].describe().T

## 7. 特征与目标的相关性 / Feature Correlation with Target

In [ ]:
# 计算各特征与目标变量的 Pearson 相关系数
# Compute Pearson correlation of each feature with the target
numeric_cols = feature_types["continuous"] + feature_types["count"] + feature_types["binary"]
full_corr = X[numeric_cols].copy()
full_corr["Target"] = y
target_corr = full_corr.corr()["Target"].drop("Target").sort_values(ascending=False)

# 绘制相关性水平条形图 / Plot horizontal bar chart of correlations
fig, ax = plt.subplots(figsize=(10, 8))
colors_bar = ["#e74c3c" if v < 0 else "#2ecc71" for v in target_corr.values]
target_corr.plot(kind="barh", ax=ax, color=colors_bar, edgecolor="white")
ax.set_title("Feature Correlation with Target / 特征与目标相关性", fontsize=14, fontweight="bold")
ax.set_xlabel("Pearson Correlation")
ax.axvline(x=0, color="gray", linestyle="--", linewidth=0.8)
plt.tight_layout()
plt.savefig(os.path.join(project_root, "data", "plots", "02_feature_target_correlation.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. 总结与洞察 / Summary & Insights

### 预处理步骤 / Preprocessing Steps Applied
1. **缺失值 / Missing values**: 未发现缺失值，数据集完整
2. **特征编码 / Feature encoding**: 8 个名义型特征经 one-hot 编码（如婚姻状态、课程、申请模式），维度从 36 扩展至约 220
3. **标准化 / Standardization**: 连续型和计数型特征标准化（零均值、单位方差）；二值型特征保持不变

### 关键发现 / Key Observations
- 数据集共 **4,424 个样本**、**36 个原始特征**
- **类别不平衡**: Graduate（49.9%）> Dropout（32.1%）> Enrolled（18.0%），模型可能需要类别权重或 SMOTE 处理
- **课程表现指标**（通过学分、成绩）与目标变量强相关，是最具预测力的特征
- **学费缴纳状态**也是重要预测因子——拖欠学费的学生更可能退学
- **宏观经济特征**（GDP、通胀率、失业率）相关性较弱，对预测贡献有限
- **奖学金获得者**倾向于毕业，**负债学生**倾向于退学

In [ ]:
# 保存处理后的数据供后续任务使用 / Save processed data for downstream tasks
X_scaled.to_csv(os.path.join(project_root, "data", "features_processed.csv"), index=False)
y.to_frame().to_csv(os.path.join(project_root, "data", "targets_processed.csv"), index=False)
print("Processed data saved / 处理后数据已保存至 data/features_processed.csv")